In [24]:
import pickle
import pandas as pd
import numpy as np

In [25]:
from tqdm import tqdm
from scipy.optimize import minimize
from functions.fitting import create_nll_features, sigmoid

def fitting_choices(preprocessed, models,exp):

    model_fits = []
    subj_list = preprocessed['subjID'].unique().tolist()
    
    for subj in tqdm(subj_list):
        trials = preprocessed[preprocessed['subjID']== subj]
        for model_name, feature_names in models.items():
            feature_weights = [0]*(len(feature_names))
            nll_func = create_nll_features(trials, feature_names, "edge", fit_randchoose=False)
            min_res = minimize(nll_func,feature_weights,method = 'SLSQP')
            res = dict(zip(feature_names + ["randchoose_logit"], min_res.x))
            # res['randchoose'] = sigmoid(res['randchoose_logit'])
            res['model'] = model_name
            res['nparam'] = len(feature_names)
            res['subjID'] = subj
            res['min_success'] = min_res.success
            res['min_res'] = min_res
            res['nll'] = min_res.fun
            
            if exp >=2:
                res['group'] = list(trials.group)[0]
                if exp == 3:
                    res['condition'] = list(trials.condition)[0]
                    res['subjectId'] = list(trials.subjectId)[0]
                    
            model_fits.append(res)
            
    df_fits= pd.DataFrame(model_fits)
    for i, row in df_fits.iterrows():
        df_fits.at[i,'BIC'] = np.log(5)*row.nparam + 2*row.nll
        
    return df_fits

In [26]:
#exp1
preprocessed = pd.read_pickle('data/preprocessed/exp1/preprocessed_exp1.pkl')

models = {
    "Level": ["feature_levels"],
    "Reward": ["feature_reward_sum"],
    "Level,Reward": ["feature_levels", "feature_reward_sum"],
    "OBM": ["OBM_AU"],
    "NBM": ["NBM_AU"],
    "POM": ["POM_AU"],
}

df_fits = fitting_choices(preprocessed, models,exp = 1)
# df_fits.to_pickle('data/preprocessed/exp1/df_fits_exp1.pkl')

100%|██████████| 99/99 [03:48<00:00,  2.30s/it]


In [27]:
#exp 2
preprocessed = pd.read_pickle('data/preprocessed/exp2/preprocessed_exp2.pkl')
models = {
    "Reward": ["feature_reward_sum"],
    "OBM": ["U_obm"],
}
df_fits = fitting_choices(preprocessed, models,exp = 2)
# df_fits.to_pickle('data/preprocessed/exp2/df_fits_exp2.pkl')

100%|██████████| 253/253 [01:26<00:00,  2.91it/s]


In [28]:


#exp 3
preprocessed = pd.read_pickle('data/preprocessed/exp3/preprocessed_exp3.pkl')
models = {
    "Reward": ["feature_reward_sum"],
    "OBM": ["U_obm"],
}
df_fits = fitting_choices(preprocessed, models,exp = 3)
# df_fits.to_pickle('data/preprocessed/exp3/df_fits_exp3.pkl')

100%|██████████| 720/720 [03:14<00:00,  3.71it/s]
